# Data preparation

## Imports

In [1]:
1 + 1

2

In [2]:
import os
import sys

sys.path.append("..")

In [3]:
import numpy as np
import pandas as pd
import dill

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [4]:
# Comment next two lines to run in collab
%load_ext autoreload
%autoreload 2

from rl_trading.features.data_processing import (
    create_reverse_fx_tickers,
)

---
## Load raw historical data

In [5]:
# comment for collab
data_folder = "C:\\Users\\Ivan\\rl_trading\\data\\"
# data_folder = ""

In [6]:
historical_data = pd.read_parquet(f"{data_folder}FX_data.parquet.gzip")

In [7]:
for col in historical_data.columns:
    if col in ["ccy", "timestamp"]:
        continue
    historical_data[col] = historical_data[col].astype(float)

In [8]:
historical_data["ccy"] = historical_data["ccy"].str.upper()

In [9]:
# historical_data = historical_data.loc[historical_data["ccy"].isin(["EURUSD"]), :]

In [10]:
historical_data["ccy"].unique()

array(['EURJPY', 'EURUSD', 'SGDJPY', 'USDJPY', 'USDSGD'], dtype=object)

---
## Add returns

In [11]:
historical_data["date"] = historical_data["timestamp"].dt.date

for lag in [1, 2, 10, 20, 30, 60, 120]:
    historical_data[f"ret_{lag}"] = historical_data.groupby(["ccy", "date"])["close"].pct_change(1).fillna(0)
    historical_data[f"ret_{lag}_sq"] = historical_data[f"ret_{lag}"] ** 2

historical_data = historical_data.drop(columns=["date"])

In [12]:
historical_data.shape

(3499125, 38)

---
## Split into 3 parts

Leave first 3 years to train scaler

In [21]:
historical_data_scaler_train = historical_data.loc[
    historical_data["timestamp"] < "2022-06-01 00:00:00", :
]

Take next 2 years for training

In [22]:
historical_data_train = historical_data.loc[
    (historical_data["timestamp"] >= "2022-06-01 00:00:00")
    & (historical_data["timestamp"] < "2024-06-01 00:00:00"),
    :,
]

Leave last month as validation set

In [23]:
historical_data_validate = historical_data.loc[
    (historical_data["timestamp"] >= "2024-06-01 00:00:00"), :
]

---
## Set close prices aside for state

In [24]:
def extract_close_prices(historical_data: pd.DataFrame) -> dict:
    """
    Extract close prices and transform to dict
    """
    historical_data = (
        pd.pivot_table(
            data=historical_data, index="timestamp", columns="ccy", values="close"
        )
        .ffill()
        .dropna()
    )

    historical_data = create_reverse_fx_tickers(historical_data)
    historical_data = historical_data.to_dict(orient="index")

    historical_data = {str(k): v for k, v in historical_data.items()}
    return historical_data

In [25]:
historical_prices_train = extract_close_prices(historical_data_train)
historical_prices_validate = extract_close_prices(historical_data_validate)

---
## Train and apply StandardScaler

In [27]:
def make_wide_data(historical_data: pd.DataFrame):
    pivoted = historical_data.pivot(index='timestamp', columns='ccy')
    pivoted.columns = [f"{ccy}_{col}" for col, ccy in pivoted.columns]
    return pivoted

In [28]:
historical_data_scaler_train = make_wide_data(historical_data_scaler_train)
historical_data_train = make_wide_data(historical_data_train)
historical_data_validate = make_wide_data(historical_data_validate)

In [32]:
normal_scaler = StandardScaler().fit(historical_data_scaler_train)

In [34]:
def apply_scaler(normal_scaler: StandardScaler, hist_data: pd.DataFrame):
    features = normal_scaler.transform(hist_data)
    features = pd.DataFrame(features, index=hist_data.index)
    return features.ffill().copy()

In [37]:
historical_data_scaler_train_scaled = apply_scaler(
    normal_scaler, historical_data_scaler_train
)
historical_data_train_scaled = apply_scaler(normal_scaler, historical_data_train)
historical_data_validate_scaled = apply_scaler(normal_scaler, historical_data_validate)

---
## Train and apply PCA

In [53]:
historical_data_scaler_train_scaled.shape

(359489, 180)

In [41]:
pca_decomposition = PCA(n_components=27).fit(
    historical_data_scaler_train_scaled.dropna()
)

Leaving those that explain more than 0.5% of variance

Alternative approach is to leave the ones with Eigen value(`explained_variance_`) higher than 1)

In [48]:
pca_decomposition.explained_variance_

array([22.38020493, 20.82477736, 19.76018145, 17.16301554, 14.24500058,
        9.98568955,  9.01600766,  8.49976357,  7.0103416 ,  6.25733032,
        5.43766758,  4.35123823,  3.26394596,  3.07012741,  2.84834531,
        2.56223665,  2.3198911 ,  2.15110412,  2.02714356,  1.64817537,
        1.61188456,  1.23956784,  1.20017087,  1.11493476,  1.05287203,
        0.96327439,  0.91459681])

In [49]:
pca_decomposition.explained_variance_ratio_

array([0.12223407, 0.11373878, 0.10792428, 0.09373932, 0.07780199,
       0.05453889, 0.04924277, 0.0464232 , 0.03828842, 0.03417569,
       0.02969893, 0.02376518, 0.01782671, 0.01676813, 0.01555682,
       0.01399418, 0.01267056, 0.0117487 , 0.01107166, 0.00900185,
       0.00880364, 0.00677015, 0.00655498, 0.00608944, 0.00575048,
       0.00526112, 0.00499526])

In [50]:
sum(pca_decomposition.explained_variance_ratio_)

np.float64(0.9444351824403768)

In [51]:
def apply_pca(pca_decomposition: PCA, hist_data: pd.DataFrame):
    features = pca_decomposition.transform(hist_data)
    final_dict = {}
    for i, date in enumerate(hist_data.index.to_list()):
        final_dict[str(date)] = features[i, :]
    return final_dict

In [52]:
historical_data_train_final = apply_pca(pca_decomposition, historical_data_train_scaled)
historical_data_validate_final = apply_pca(
    pca_decomposition, historical_data_validate_scaled
)

---
## Save results

In [54]:
final_data = {
    "train_prices": historical_prices_train,
    "train_data": historical_data_train_final,
    "validate_prices": historical_prices_validate,
    "validate_data": historical_data_validate_final,
}

In [55]:
data_folder = "C:\\Users\\Ivan\\rl_trading\\data\\"

In [56]:
%%time
with open(f"{data_folder}preprocessed_data.pkl", "wb") as f:
    dill.dump(final_data, f, dill.HIGHEST_PROTOCOL)

CPU times: total: 1min 12s
Wall time: 1min 12s
